In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Model-based Value methods


We will study two common Model-based, Dynamic Programming type methods
- Value Iteration
- Policy Iteration

and their associated Bellman equations.



## Value iteration method

The simplest method is to
- iteratively update the Value function
    - until convergence
- derive the *final* Policy from the Value function
    - choose the action leading to highest return
    - based on the Value function

The estimate of the value of a state $\state$ is improved iteratively.

Let 
$$
\statevalfun_{k}(\state) 
$$

denote the *estimated* value of state $\state$ after $k$ iterations.

In iteration $k+1$, the value of *each* state $\state \in \States$ is updated via the Bellman update


$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc \statevalfun_k(\state')\bigr).
$$

where 
$$
\rew(\state,\act,\state')
$$

denotes the reward returned by the environment when action $\act$ in state $\state$
results in successor state $\state'$

- the fixed point of this update is the Bellman *optimality* equation
$$
\statevalfun_*(\state) = \max{\act} \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc \statevalfun_*(\state')\bigr)
$$
- value iteration repeats the update until the table stops changing


That is
- $\statevalfun_{k+1}(\state)$ 
- uses the *previous iteration's $(k)$* estimate $$\statevalfun_k(\state')$$  
of each successor state $\state'$
- and chooses the action leading to highest return

The backup in each iteration
- moves information about potential future (successor) states
- to the current state


For Frozen Lake
- the only non-zero reward is obtained at terminal state (the Gift) G

Thus, in iteration $1$
- the only non-zero information transfered is from G to its predecessor, square $14$

This information can only diffuse to more distant predecessors 
- through multiple iterations

The previous iteration's estimate of the successors of $\state$
$$\statevalfun_k(\state')$$  

in turn, are based on the iteration $(k-1)$ values of the successors of $\state'$

So we need $i$ iterations
- in order for information of states $i$ steps ahead
- to reach state $\state$

Hence the need to repeat until convergence.


**Value iteration on Frozen Lake** (non-slippery, discount $0.9$)

<img src="images/value_iteration_strip.png" width=100%>

- sweep 1: only the square next to the goal gains value
- each further sweep moves information one square back from G
- after $6$ sweeps the values stop changing (sweep $7$ confirms that nothing changes): they are the optimal values, and the arrows are the optimal policy
- every square is updated from the *previous* sweep's table (synchronous)
- worked backup, square $10$ at sweep $2$: down reaches square $14$, worth $1.00$ after sweep $1$, so $\statevalfun_2(10) = 0 + 0.9 \times 1.00 = 0.90$ (right is a hole)
- sweeps $4$ and $5$ (not shown) give squares $2$ and $8$ the value $0.73$, then squares $1$, $3$ and $4$ the value $0.66$
- the pseudo-code on the next slides overwrites $\statevalfun$ in place instead; the result is the same


**Subtlety**

Prior to convergence
- $\statevalfun_k(\state)$ is a *non-random* variable
    - $\transp(\state', \rew \mid \state, \act)$ is *known* for all $\state, \act$ since method is model-based
- the estimate is *not yet correct*
    - its error comes from stopping early, not from randomness

We will subsequently contrast this to the iterative estimates produced by model-free methods.


**Pseudo code for Value Iteration**

Here is some pseudo-code; the book's $p(s', r \mid s, a)$ is our $\transp(\state', \rew \mid \state, \act)$ and its $\gamma$ is our $\disc$

<br>

<table>
    <center><strong>Value iteration</strong></center>
    <tr>
        <img src="images/RL_value_iteration_alg.png" width=90%>
    </tr>
    

</table>

Attribution: http://incompleteideas.net/book/RLbook2020.pdf#page=105

<table>
    <center><strong>Value iteration</strong></center>

    Initialize value function V(s) arbitrarily (e.g., zero for all states; terminal states keep V = 0 and are never updated)

    Repeat:
        delta = 0
        For each state s:
            old_value = V(s)
            V(s) = max over a [ sum over s' [ P(s' | s, a) * ( r(s, a, s') + γ * V(s') ) ] ]
            delta = max(delta, |old_value - V(s)|)

        Until delta < threshold

    # Derive policy after value function converges
    For each state s:
        π(s) = argmax over a [ sum over s' [ P(s' | s, a) * ( r(s, a, s') + γ * V(s') ) ] ]

    Return π, V
    
</table>


## Policy iteration method

The Value Iteration method has one expensive step
- the $\max{\act}$ is a search over all possible actions $\act \in \Actions$

$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc \statevalfun_k(\state')\bigr).
$$

Policy iteration moves that cost elsewhere: it replaces the max inside every sweep by repeated full evaluations of one policy.


Policy Iteration is another way to reach the optimal $\statevalfun$ and $\pi_*$: it avoids the $\max{\act}$ inside every sweep, but pays for fully evaluating each policy.

Rather than updating the Policy once (after Value function convergence)
- we introduce a method that periodically updates the Policy.


*Policy iteration* is an algorithm that improves $\pi_p$ to $\pi_{p+1}$ by alternating two steps
during round $p$

The algorithm alternates between
- Policy evaluation
    - compute $\statevalfun_{\pi_p}$, the value of the *current* policy $\pi_p$
- Policy improvement
    - make $\pi_{p+1}$ greedy with respect to $\statevalfun_{\pi_p}$

It uses the Bellman update

$$
\statevalfun_{\pi,k+1}(\state) =  \sum_{\state'} \transp(\state' \mid \state, \pi(\state))\bigl(\rew(\state,\pi(\state),\state') + \disc \statevalfun_{\pi,k}(\state')\bigr).
$$

where
- we add an explicit subscript $\pi$
- to the value function
$$
\statevalfun_{\pi,k+1}(\state)
$$
- to indicate the dependence of the update on the *current policy* $\pi$

Rather than
- evaluating *all* actions $\act \in \Actions$
- it chooses a single action $\pi(\state)$ according to the current policy

**Policy iteration on Frozen Lake**

<img src="images/policy_iteration_rounds.png" width=100%>

- start from *always down*; each round evaluates the current policy, then improves it greedily
- $4$ rounds reach the optimal policy; orange arrows mark the actions that changed
- the numbers are the value of the *current* policy, so they differ from the optimal values until the last round


Here is some pseudo-code

<br>

<table>
    <center><strong>Policy iteration</strong></center>
    <tr>
        <img src="images/RL_policy_iteration_alg.png" width=90%>
    </tr>
    

</table>

Attribution: http://incompleteideas.net/book/RLbook2020.pdf#page=102

**Pseudo code for Policy Iteration**

<table>
    <center><strong>Policy iteration</strong></center>

    Initialize policy π arbitrarily (e.g., an arbitrary action in each state)
    Initialize value function V(s) arbitrarily (e.g., zero for all states; terminal states keep V = 0)

    Repeat:
        # Policy Evaluation (compute V for current policy π)
        Repeat:
            For each state s:
                V(s) = sum over s' [ P(s' | s, π(s)) * ( r(s, π(s), s') + γ * V(s') ) ]
            Until V(s) converges (changes smaller than threshold)

        # Policy Improvement (update policy based on current V)
        policy_stable = True
        For each state s:
            old_action = π(s)
            π(s) = argmax over a [ sum over s' [ P(s' | s, a) * ( r(s, a, s') + γ * V(s') ) ] ]
            if old_action != π(s):
                policy_stable = False

    Until policy_stable is True

    Return π, V
    
</table>

Both Value Iteration and Policy Iteration converge to an optimal policy (they may differ only where actions tie).

The advantage of alternating between Policy Evaluation and Policy Improvement
- fewer rounds to converge (each round costs more) 
    - the Value function under the current policy is fully evaluated
    - before the Policy is updated

On Frozen Lake (computed)
- value iteration needs $7$ sweeps (the last confirms that nothing changes)
- policy iteration from "always down" needs $4$ rounds, but $11$ evaluation sweeps in total
- fewer rounds does not mean less work


## From a state value function to an action value function

With the model, a state value function gives a policy by one-step lookahead (the policy slide earlier in this session)
- try each action, weigh the possible next states with the model $\transp$
- pick the action with the best reward plus discounted next value

Without the model that lookahead cannot be computed.

Storing a value for each state-*action* pair removes the lookahead
- the best action is read off directly


Define
an *action value function* $\actvalfun_\pi(\state, \act)$
$$
\actvalfun_{\pi}: \States \times \Actions \to \Reals
$$

to map a state/action pair $(\state, \act)$ to the expected return of taking action $\act$ in state $\state$ and following $\pi$ afterwards.

- with $\actvalfun$, the best action is simply $\argmax{\act} \actvalfun(\state, \act)$
- no model is needed for that last step (computing $\actvalfun$ *from* $\statevalfun$, as on the next slide, still uses the model; later sessions learn $\actvalfun$ directly from experience)
- $\actvalfun_*$ is the same under optimal play, and $\statevalfun_*(\state) = \max{\act}\,\actvalfun_*(\state, \act)$
- the Bellman equation for the action value
$$
\actvalfun_\pi(\state, \act) = \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc \sum_{\act'} \pi(\act' \mid \state')\,\actvalfun_\pi(\state', \act')\bigr)
$$
- from *sampled* transitions (state, action, reward, next state) the same update can be made without $\transp$: that is the next session


Re-write the Value function backup equation
$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc \statevalfun_k(\state')\bigr).
$$

as

$$
\statevalfun_{k+1}(\state) = \max{\act} \actvalfun_k(\state, \act)
$$

where

$$
\actvalfun_k(\state,\act) = \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc \statevalfun_k(\state')\bigr)
$$

## Key Differences Between Value Iteration and Policy Iteration

Here is a comparison of the Value and Policy Iteration methods.

| Feature              | Value Iteration                                                      | Policy Iteration                                    |
|:----------------------|:---------------------------------------------------------------------|:-----------------------------------------------------|
| Approach             | Updates value function until convergence                            | Alternates between value evaluation and improvement |
| Convergence          | When value function $\statevalfun(\state)$ stabilizes                               | When policy $\pi(\state)$ stops improving                |
| Computational Cost   | One sweep, with a max over actions in every state                                 | Per round: a full evaluation of the current policy (several sweeps), then one argmax over actions                   |
| Speed                | Requires more iterations                                            | Fewer rounds; each round costs more                      |
| Policy Output        | Extracted after value convergence                                   | Updated during each iteration                       |






# When the table is too big

- a table needs one entry per state (or per state and action)
- Microduck's state is $61$ continuous numbers, so a neural network $\statevalfun_\theta(\state)$ replaces the table
    - learning such a network from experience is a later session (DQN)
- note: this is not a *learned model* of the environment, which is a different idea we do not cover here


# What we established, and what it sets up

- **Established**
    - the Bellman equation relates the value of a state to the values of its successors
    - with a known model, value iteration and policy iteration sweep to the optimal values and policy
    - a policy needs a one-step lookahead (which uses the model) or action values
- **Sets up**
    - next session: the model is unknown, so values are learned from sampled experience (Monte Carlo, temporal difference, Q-learning)
    - later: a neural network replaces the table when the state space is large


In [2]:
print("Done")

Done
